# 07 — Validation-selected RUL calibration

**Goal:** Test whether subtracting a *validation-selected* number of cycles from the Iteration 3 Random Forest predictions reduces near-failure overestimation for FD002 and FD004 while limiting deterioration in overall error.

**Leakage control:** Calibration choices use **only engine-separated validation predictions**. Official test predictions are opened **after** the offsets have been selected. Models and RUL caps are fixed using `iteration5_model_choices.csv` (selected in Iteration 3). This notebook does **not** retrain models.

**Important:** `predicted_RUL` is a capped-horizon estimate, not an exact lifetime beyond the chosen cap. Calibration cannot solve this limitation. All error definitions below use `prediction − truth`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Compatible with notebooks/ or repository root as working directory.
candidates = [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]
required = ['iteration5_validation_predictions.csv', 'iteration5_model_choices.csv',
            'iteration5_official_test_predictions.csv']
search_dirs = []
for base in candidates:
    search_dirs.extend([base, base/'results', base/'results'/'iteration5',
                        base/'results'/'robustness', base/'results'/'model_improvements'])

def locate(filename):
    for folder in search_dirs:
        if (folder / filename).is_file():
            return folder / filename
    # Look in existing project results folders only.
    for base in candidates[:2]:
        if base.is_dir():
            hits = list(base.glob('results/**/' + filename))
            if hits: return hits[0]
    raise FileNotFoundError(f'{filename}: move into a results folder or beside this notebook')

validation_path = locate(required[0]); choices_path = locate(required[1])
validation = pd.read_csv(validation_path)
choices = pd.read_csv(choices_path)
output = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'results' / 'calibration'
output.mkdir(parents=True, exist_ok=True)
print('Validation:', validation_path)
print('Model choices:', choices_path)
print('Output:', output)
display(choices[['dataset','variant','cap']])

## 1. Define fair comparison metrics

- **Raw RMSE:** measured against uncapped, actual RUL (cycles); this includes unavoidable high-RUL errors for capped models.
- **Near-failure MAE:** absolute error for engines with actual RUL ≤ 30 cycles.
- **Near-failure signed bias:** mean(prediction − actual), where positive is optimistic.
- **Near-failure optimistic fraction:** proportion for which prediction exceeds truth.
- **Capped-target RMSE:** supplementary, measured against min(actual RUL, model cap).

Offset candidates are **0–15 cycles in one-cycle steps**. We select the smallest near-failure MAE among offsets whose validation **raw RMSE is no more than 5% worse** than the zero-offset result. The 5% tolerance is a stated engineering trade-off, *not* a proven operational safety threshold.

In [ ]:
NEAR_LIMIT = 30
OFFSETS = np.arange(0, 16, 1)
MAX_RMSE_INCREASE = 0.05

def evaluate(actual, prediction, cap):
    y = np.asarray(actual, float)
    p = np.asarray(prediction, float)
    good = np.isfinite(y) & np.isfinite(p)
    y, p = y[good], p[good]
    if not len(y): raise ValueError('No valid predictions')
    error = p - y
    near = y <= NEAR_LIMIT
    result = dict(n=int(len(y)), n_near=int(near.sum()),
                  raw_MAE=float(np.abs(error).mean()),
                  raw_RMSE=float(np.sqrt(np.mean(error**2))),
                  raw_bias=float(error.mean()),
                  capped_RMSE=float(np.sqrt(np.mean((p-np.minimum(y,cap))**2))))
    result.update(near_MAE=float(np.abs(error[near]).mean()) if near.any() else np.nan,
                  near_bias=float(error[near].mean()) if near.any() else np.nan,
                  near_over_fraction=float((error[near]>0).mean()) if near.any() else np.nan)
    return result

for col in ['dataset','seed','variant','cap','engine_id','actual_RUL','predicted_RUL']:
    if col not in validation: raise ValueError(f'Missing validation column: {col}')
for col in ['dataset','variant','cap']:
    if col not in choices: raise ValueError(f'Missing model-choice column: {col}')
assert not choices.dataset.duplicated().any(), 'One chosen model per dataset expected'
assert validation[['dataset','seed','variant','cap','engine_id']].duplicated().sum() == 0

## 2. Compare offsets across the three validation splits

The notebook computes errors within each engine-separated validation split first, then takes the mean across splits. This avoids overstating certainty by treating the same engine appearing across different splits as independent evidence.

In [ ]:
rows = []
for choice in choices.itertuples(index=False):
    selected = validation[(validation.dataset == choice.dataset) &
                          (validation.variant == choice.variant) &
                          (validation.cap == choice.cap)].copy()
    if selected.empty: raise ValueError(f'No validation records for {choice.dataset}')
    for seed, group in selected.groupby('seed'):
        for offset in OFFSETS:
            pred = np.maximum(0, group.predicted_RUL.to_numpy(float) - offset)
            m = evaluate(group.actual_RUL, pred, float(choice.cap))
            rows.append(dict(dataset=choice.dataset, variant=choice.variant,
                             cap=choice.cap, seed=seed, offset=int(offset), **m))
by_split = pd.DataFrame(rows)
metric_cols = ['raw_MAE','raw_RMSE','raw_bias','capped_RMSE',
               'near_MAE','near_bias','near_over_fraction']
summary = by_split.groupby(['dataset','variant','cap','offset'],as_index=False).agg(
    **{k:(k,'mean') for k in metric_cols},
    raw_RMSE_sd=('raw_RMSE','std'), near_MAE_sd=('near_MAE','std'),
    n_near_mean=('n_near','mean'))
by_split.to_csv(output/'calibration_validation_by_split.csv',index=False)
summary.to_csv(output/'calibration_validation_grid.csv',index=False)
display(summary[['dataset','offset','raw_RMSE','near_MAE','near_bias','near_over_fraction']].head(10))

In [ ]:
decisions=[]
for dataset, group in summary.groupby('dataset'):
    zero = group.loc[group.offset.eq(0)].iloc[0]
    acceptable = group.loc[group.raw_RMSE <= zero.raw_RMSE*(1+MAX_RMSE_INCREASE)+1e-10]
    # Stable, explicit tie-break: lower near MAE, lower raw RMSE, smaller offset.
    best = acceptable.sort_values(['near_MAE','raw_RMSE','offset']).iloc[0]
    decisions.append(dict(dataset=dataset, variant=best.variant, cap=int(best.cap),
                          selected_offset=int(best.offset), baseline_val_raw_RMSE=zero.raw_RMSE,
                          calibrated_val_raw_RMSE=best.raw_RMSE,
                          baseline_val_near_MAE=zero.near_MAE,
                          calibrated_val_near_MAE=best.near_MAE,
                          calibrated_val_near_bias=best.near_bias,
                          max_allowed_RMSE_increase_pct=100*MAX_RMSE_INCREASE))
decision_table = pd.DataFrame(decisions).sort_values('dataset')
decision_table.to_csv(output/'calibration_selected_offsets.csv',index=False)
display(decision_table)

## 3. Visualise validation trade-offs

Each point is a possible offset. The vertical dashed line shows the maximum allowed validation raw RMSE based on the unadjusted model. The star marks the selected offset.

In [ ]:
for dataset, part in summary.groupby('dataset'):
    chosen = decision_table.set_index('dataset').loc[dataset]
    base = part[part.offset.eq(0)].iloc[0]
    fig, ax = plt.subplots(figsize=(7,5))
    scatter = ax.scatter(part.raw_RMSE,part.near_MAE,c=part.offset,cmap='viridis',s=65)
    ax.scatter(chosen.calibrated_val_raw_RMSE,chosen.calibrated_val_near_MAE,
               marker='*',s=270,color='crimson',label=f"Selected −{chosen.selected_offset} cycles")
    ax.axvline(base.raw_RMSE*(1+MAX_RMSE_INCREASE),color='gray',linestyle='--',
               label='5% RMSE deterioration limit')
    for row in part.itertuples():
        if row.offset in (0,5,10,15): ax.annotate(str(row.offset),(row.raw_RMSE,row.near_MAE),xytext=(4,4),textcoords='offset points',fontsize=9)
    fig.colorbar(scatter,ax=ax,label='Subtracted offset (cycles)')
    ax.set(title=f'{dataset}: calibration trade-off (validation)',
           xlabel='Raw RUL RMSE (cycles; lower is better)',ylabel='Near-failure MAE (cycles; lower is better)')
    ax.legend(fontsize=8);fig.tight_layout()
    fig.savefig(output/f'{dataset.lower()}_validation_tradeoff.png',dpi=220)
    fig.savefig(output/f'{dataset.lower()}_validation_tradeoff.pdf')
    plt.show()

## 4. Evaluate once on the official test set

**No tuning after looking at these results.** This cell loads the official predictions and applies only the already selected validation offsets. The original `predicted_unadjusted` is retained as the common reference (do not use the Iteration 3 fixed-five adjusted column). Results reflect this post-processing strategy—not a newly trained model.

In [ ]:
test_path = locate('iteration5_official_test_predictions.csv')
test = pd.read_csv(test_path)
for col in ['dataset','engine_id','actual_RUL','predicted_unadjusted']:
    if col not in test: raise ValueError(f'Missing test column: {col}')
assert not test[['dataset','engine_id']].duplicated().any()
metrics=[]; predictions=[]
for d in decision_table.itertuples(index=False):
    s=test[test.dataset.eq(d.dataset)].copy()
    if s.empty: raise ValueError(f'No test records for {d.dataset}')
    s['selected_offset']=d.selected_offset
    s['predicted_calibrated']=np.maximum(0,s.predicted_unadjusted-d.selected_offset)
    s['error_unadjusted']=s.predicted_unadjusted-s.actual_RUL
    s['error_calibrated']=s.predicted_calibrated-s.actual_RUL
    predictions.append(s)
    for label,column in [('Unadjusted','predicted_unadjusted'),('Validation-calibrated','predicted_calibrated')]:
        metrics.append(dict(dataset=d.dataset,method=label,offset=d.selected_offset if label!='Unadjusted' else 0,
                            **evaluate(s.actual_RUL,s[column],d.cap)))
final_test=pd.DataFrame(metrics)
final_preds=pd.concat(predictions,ignore_index=True)
final_test.to_csv(output/'calibration_official_test_metrics.csv',index=False)
final_preds.to_csv(output/'calibration_official_test_predictions.csv',index=False)
display(final_test[['dataset','method','offset','raw_RMSE','near_MAE','near_bias','near_over_fraction','n_near']].round(3))

## 5. Report figures: error by remaining-life band and near-failure bias

In [ ]:
bounds=[-0.001,30,60,100,150,np.inf]
labels=['0–30','31–60','61–100','101–150','151+']
final_preds['rul_band']=pd.cut(final_preds.actual_RUL,bins=bounds,labels=labels)
for dataset, sub in final_preds.groupby('dataset'):
    band=sub.groupby('rul_band',observed=False)[['error_unadjusted','error_calibrated']].agg(
        lambda x: np.abs(x).mean() if len(x) else np.nan)
    fig,ax=plt.subplots(figsize=(8,4.5))
    xpos=np.arange(len(band));width=0.36
    ax.bar(xpos-width/2,band.error_unadjusted,width,label='Unadjusted')
    ax.bar(xpos+width/2,band.error_calibrated,width,label='Validation-calibrated')
    ax.set_xticks(xpos,band.index.astype(str));ax.set(xlabel='Actual RUL (cycles)',ylabel='Mean absolute error (cycles)',
          title=f'{dataset}: errors by actual RUL band')
    ax.legend();fig.tight_layout()
    fig.savefig(output/f'{dataset.lower()}_error_by_rul_band.png',dpi=220)
    fig.savefig(output/f'{dataset.lower()}_error_by_rul_band.pdf')
    plt.show()

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,key,title in zip(axes,['near_bias','near_over_fraction'],['Near-failure signed bias','Near-failure optimistic fraction']):
    pivot=final_test.pivot(index='dataset',columns='method',values=key)
    x=np.arange(len(pivot));w=0.35
    for j,method in enumerate(['Unadjusted','Validation-calibrated']):
        ax.bar(x+(j-0.5)*w,pivot[method],w,label=method)
    ax.set_xticks(x,pivot.index)
    ax.set(title=title,ylabel='Cycles' if key=='near_bias' else 'Fraction of near-failure engines')
    if key=='near_bias':ax.axhline(0,color='gray',linewidth=1)
axes[0].legend(fontsize=8)
fig.tight_layout();fig.savefig(output/'near_failure_calibration_comparison.png',dpi=220)
fig.savefig(output/'near_failure_calibration_comparison.pdf');plt.show()

## 6. Engineering interpretation and limitations

1. **Positive near-failure signed bias** means the prediction is optimistic when failure is close; reducing it may be more valuable than reducing global error alone.
2. **A smaller optimistic fraction does not establish safety.** We have no maintenance-cost model, calibrated failure probability or real aircraft validation.
3. A capped RUL model **cannot recover true long-horizon lifetimes through subtracting a constant**. Interpret results ≥ cap as horizon-limited.
4. Validation splits may overlap in engines across seeds. The mean and standard deviation summarise sensitivity to the split, not independent replicated experiments or a formal confidence interval.
5. If the official test metrics worsen, **report that outcome**. Do not change the offset after examining the official test results.
6. Confirm the same engine identifiers and evaluation horizons are used when comparing with Iteration 2.

**Suggested report sentence:** “A constant downward adjustment was selected using repeated engine-separated validation, subject to a maximum 5% deterioration in validation raw-RUL RMSE. The fixed adjustment was then evaluated on the official test set to quantify the trade-off between near-failure optimism and overall prediction accuracy.”

In [ ]:
print('Files saved to:',output.resolve())
print('Selected offset(s):')
display(decision_table[['dataset','selected_offset','baseline_val_raw_RMSE','calibrated_val_raw_RMSE',
                        'baseline_val_near_MAE','calibrated_val_near_MAE']].round(2))
print('Test metrics:')
display(final_test[['dataset','method','raw_RMSE','near_MAE','near_bias','near_over_fraction']].round(2))